# Buyscape Nigeria second-hand market: a first look

One row per advert for used goods in Nigeria: its link, asking price in naira, city, condition, the date it was read and our photo check. No photos and no sellers' words are included; follow each link to see the advert.

The `test` rows of the Testify benchmark show only their links, so they stay an honest test.

Jiji adverts are used with the permission of Jiji Nigeria (jiji.ng), granted on 4 Nov 2026, on our terms: links and our own labels only, never Jiji photos or seller contact details. Credit: Jiji Nigeria, the data source.

The benchmark built from these adverts, and how a fine-tuned 2B model compares with Claude Sonnet 5 and Nemotron on it, is at [huggingface.co/datasets/godzilla1215/testify](https://huggingface.co/datasets/godzilla1215/testify).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

# Wherever Kaggle mounts the dataset.
DATA = os.path.dirname(glob.glob('/kaggle/input/**/adverts.csv', recursive=True)[0])
df = pd.read_csv(f'{DATA}/adverts.csv')
print(f'{len(df):,} adverts, {df.city.nunique()} cities, {df.kind.nunique()} kinds')
df.head()

## What each advert is used for

- **training**: adverts for teaching a small model to read and price items
- **benchmark** and **held back**: the Testify benchmark and the adverts its photo check left out
- **new stock** and **shop price**: new prices from dealers and shops, for valuing items whose used market is thin

In [ ]:
df.role.value_counts()

## Where the adverts were listed

In [ ]:
df.city.fillna('not stated').value_counts().head(15).plot.barh(figsize=(8, 5), title='Adverts by city')
plt.gca().invert_yaxis()
plt.show()

## Asking prices by kind

The median asking price for the 20 most common kinds of second-hand item, leaving out new stock and shop prices.

In [ ]:
used = df[df.role.isin(['training', 'benchmark', 'held back']) & df.price_ngn.notna()]
top = used.kind.value_counts().head(20).index
summary = (used[used.kind.isin(top)].groupby('kind').price_ngn
           .agg(adverts='count', median_naira='median').sort_values('median_naira'))
summary.median_naira.plot.barh(figsize=(8, 7), logx=True, title='Median asking price, naira (log scale)')
plt.show()
summary

## Price bands

In [ ]:
bands = pd.cut(used.price_ngn, [0, 1e4, 5e4, 2e5, 1e6, 5e6, float('inf')],
               labels=['under 10k', '10k to 50k', '50k to 200k', '200k to 1m', '1m to 5m', 'over 5m'])
bands.value_counts().sort_index()

## Condition and the photo check

`photo_check_identity` says whether the photos alone show what the item is: `named` (a brand and model can be read), `by_type` (the kind of item is clear) or `unknown`.

In [ ]:
print(df.condition.fillna('not stated').value_counts(), '\n')
print(df.photo_check_identity.value_counts())

## Finding reposts

The JSON lines file carries a perceptual hash for each photo. Adverts that share a photo hash are usually the same item posted again.

In [ ]:
rows = pd.read_json(f'{DATA}/adverts.jsonl', lines=True)
hashes = rows[['id', 'photo_hashes']].explode('photo_hashes').dropna()
shared = hashes.groupby('photo_hashes').id.nunique()
print(f'{(shared > 1).sum():,} photo fingerprints appear in more than one advert')